# Project Lab 1: Logs, Git Flow and Benchmarks

Notebook for **Chapter 15** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to Run a Project in This Area

## 15.2 Project: Log-Analysis Toolkit (Shell plus Python)

In [ ]:
# src/gen_logs.py
import random, sys
from datetime import datetime, timedelta

def main(path, n=2000, seed=7):
    rnd = random.Random(seed)
    urls = ["/", "/products", "/products/42", "/cart", "/checkout", "/login", "/api/orders", "/missing-page"]
    weights = [30, 25, 12, 10, 6, 8, 7, 2]
    status = {"/": [200], "/products": [200], "/products/42": [200, 404], "/cart": [200, 500],
              "/checkout": [200, 500, 502], "/login": [200, 401], "/api/orders": [200, 500],
              "/missing-page": [404]}
    t = datetime(2026, 3, 1, 9, 0, 0)
    with open(path, "w") as f:
        for _ in range(n):
            t += timedelta(seconds=rnd.randint(0, 3))
            u = rnd.choices(urls, weights)[0]
            s = rnd.choice(status[u])
            ip = f"10.0.{rnd.randint(0,3)}.{rnd.randint(1,50)}"
            f.write(f'{ip} - - [{t:%d/%b/%Y:%H:%M:%S} +0000] "GET {u} HTTP/1.1" {s} {rnd.randint(200, 5000)}\n')
        f.write("this line is damaged\n")

if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
# src/logstats.py
import re, argparse
from collections import Counter

LINE = re.compile(r'^(?P<ip>\S+) \S+ \S+ \[(?P<ts>[^\]]+)\] "(?P<method>\S+) (?P<url>\S+) [^"]*" (?P<status>\d{3}) (?P<size>\d+)$')

def parse(lines):
    bad = 0
    for line in lines:
        m = LINE.match(line.rstrip("\n"))
        if m: yield m.groupdict()
        else: bad += 1
    parse.bad = bad            # set after the generator is fully read

def summarise(path, top=3):
    status, urls, errors_by_url = Counter(), Counter(), Counter()
    total = 0
    with open(path) as f:
        for rec in parse(f):
            total += 1
            status[rec["status"]] += 1
            urls[rec["url"]] += 1
            if rec["status"].startswith("5"):
                errors_by_url[rec["url"]] += 1
    return {"total": total, "bad": parse.bad, "status": status,
            "urls": urls.most_common(top), "errors": errors_by_url.most_common(top)}

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("logfile"); ap.add_argument("--top", type=int, default=3)
    a = ap.parse_args()
    r = summarise(a.logfile, a.top)
    print(f"lines parsed: {r['total']}  skipped (bad): {r['bad']}")
    print("status counts:", dict(sorted(r["status"].items())))
    print("top URLs:");     [print(f"  {c:5d}  {u}") for u, c in r["urls"]]
    print("top 5xx URLs:"); [print(f"  {c:5d}  {u}") for u, c in r["errors"]]
    five = sum(c for s, c in r["status"].items() if s.startswith("5"))
    print(f"server error rate: {five / r['total'] * 100:.2f}%")

if __name__ == "__main__":
    main()

In [ ]:
# tests/test_logstats.py
import sys, tempfile, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from logstats import summarise

def test_small():
    text = ('1.1.1.1 - - [01/Mar/2026:09:00:01 +0000] "GET /a HTTP/1.1" 200 10\n'
            '1.1.1.1 - - [01/Mar/2026:09:00:02 +0000] "GET /a HTTP/1.1" 500 10\n'
            'garbage\n')
    with tempfile.NamedTemporaryFile("w", delete=False, suffix=".log") as f:
        f.write(text)
    r = summarise(f.name)
    assert r["total"] == 2 and r["bad"] == 1
    assert r["errors"] == [("/a", 1)]
    print("test_small ok")

if __name__ == "__main__":
    test_small()

## 15.3 Project: Git Workflow Practice Repository

## 15.4 Project: Sorting and Searching Benchmark Harness

In [ ]:
# algos.py
def bubble_sort(a):
    a = list(a); n = len(a)
    for i in range(n):
        swapped = False
        for j in range(n - 1 - i):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]; swapped = True
        if not swapped: break
    return a

def merge_sort(a):
    if len(a) <= 1: return list(a)
    mid = len(a) // 2
    l, r = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(l) and j < len(r):
        if l[i] <= r[j]: out.append(l[i]); i += 1
        else:            out.append(r[j]); j += 1
    return out + l[i:] + r[j:]

def linear_search(a, x):
    for i, v in enumerate(a):
        if v == x: return i
    return -1

def binary_search(a, x):
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == x: return mid
        if a[mid] < x: lo = mid + 1
        else: hi = mid - 1
    return -1

In [ ]:
# bench.py (main parts)
import random, time, statistics
from algos import *

def timeit(fn, *args, repeats=5):
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(*args); runs.append((time.perf_counter() - t0) * 1000)
    return statistics.median(runs)          # milliseconds

def check_correct():
    rnd = random.Random(1)
    for n in (0, 1, 2, 10, 200):
        a = [rnd.randint(0, 50) for _ in range(n)]
        assert bubble_sort(a) == merge_sort(a) == sorted(a)
    s = list(range(0, 100, 2))
    assert binary_search(s, 42) == 21 and binary_search(s, 43) == -1 and linear_search(s, 42) == 21

def table(title, rows, header):
    print(title); print(" | ".join(f"{h:>12}" for h in header))
    for r in rows: print(" | ".join(f"{c:>12}" for c in r))
    print()

def growth(rows):
    return ["-"] + [f"{rows[i][1]/rows[i-1][1]:.1f}x" for i in range(1, len(rows))]

## 15.5 Benchmark Statistics: Measure Like You Mean It

A single timing is a rumour. Your laptop is also running a browser, a backup and a clock update, so the same code takes 52 ms, then 49, then 71. To compare two programs you need several runs and a summary that ignores the occasional bad one.

In [ ]:
import statistics as st
import numpy as np
t = [50, 52, 49, 51, 120, 50, 53, 48, 51]
q1, q3 = np.percentile(t, [25, 75])
print("mean", round(st.mean(t), 1), "median", st.median(t), "min", min(t))
print("IQR", q3 - q1)

mean 58.2 median 51 min 48
IQR 2.0


In [ ]:
import time, statistics as st
import numpy as np

def bench(fn, repeats=9, min_time=0.01):
    fn()                                   # warm-up
    loops = 1
    while True:                            # find a loop count that takes at least min_time
        t = time.perf_counter()
        for _ in range(loops): fn()
        if time.perf_counter() - t >= min_time: break
        loops *= 2
    runs = []
    for _ in range(repeats):
        t = time.perf_counter()
        for _ in range(loops): fn()
        runs.append((time.perf_counter() - t) / loops)
    q1, q3 = np.percentile(runs, [25, 75])
    return {"median": st.median(runs), "iqr": q3 - q1, "min": min(runs)}

data = list(range(10_000))
def loop_sum():
    total = 0
    for x in data: total += x
    return total
def builtin_sum(): return sum(data)

a, b = bench(loop_sum), bench(builtin_sum)
print("same answer:", loop_sum() == builtin_sum())
print("builtin faster by at least 2x:", b["median"] * 2 < a["median"])
print("ranges do not overlap:", b["median"] + b["iqr"] < a["median"] - a["iqr"])

same answer: True
builtin faster by at least 2x: True
ranges do not overlap: True


## Chapter recap

## Practice set

Problem 15.1 (easy) In the sample log, 211 requests returned 500 and 41 returned 502, out of 2,000 parsed lines. What is the server error rate?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch15.html